Для начала проверим получилось доступпна ли нам видеокарта для обучения

In [1]:
import torch

print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

2.3.1+cu121
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Импортируем нужные библиотеки

In [2]:
import pandas as pd
from pathlib import Path
from PIL import Image
import shutil
from sklearn.model_selection import train_test_split
import os

Теперь мы должны привести датасет в нужный для нас формат. Разделим train на train и val. Достанем информацию из csv файлов и сделаем для каждой картинки txt файл, который будет ее описывать. 

In [3]:
data_path = Path("data")

images_path = data_path / "train"
csv_path = data_path / "train_solution_bounding_boxes (1).csv"

output_path = Path("dataset")

labels_train = output_path / "labels/train"
labels_val = output_path / "labels/val"

images_train = output_path / "images/train"
images_val = output_path / "images/val"

for p in [labels_train, labels_val, images_train, images_val]:
    p.mkdir(parents=True, exist_ok=True)

Посмотрим как выглядит csv файл:

In [4]:
df = pd.read_csv(csv_path)
df.head()

,image,xmin,ymin,xmax,ymax
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422


Мы видим координаты. Теперь мы должны достать их из csv и для каждого изображения создать отдельный txt файл такого вида:

0 0.5137481910502958 0.5520431146052631 0.17945007233727814 0.11711711710526315

Здесь первый 0 описывает класс, присутствующий на изображении, но т.к. у нас только 1 класс, то у нас в этой строке везде будет 0, другие же 4 числа - координаты, приведенные в нужный нам формат. Ниже представлена функция, которая это делает:

In [5]:
def convert(xmin, ymin, xmax, ymax, w, h):
    x_center = (xmin + xmax) / 2 / w
    y_center = (ymin + ymax) / 2 / h
    width = (xmax - xmin) / w
    height = (ymax - ymin) / h
    return x_center, y_center, width, height

Берем 20 процентов от train выборки и делаем из них валидационную выборку

In [6]:
images = df["image"].unique()

train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

Копируем изображения и создаем txt файлы нужного формата

In [7]:
def copy_images(img_list, split):
    for img_name in img_list:
        src = images_path / img_name
        dst = output_path / f"images/{split}" / img_name

        if src.exists():
            shutil.copy(src, dst)

copy_images(train_imgs, "train")
copy_images(val_imgs, "val")

In [8]:
def create_labels(img_list, split):
    for img_name in img_list:
        img_path = images_path / img_name

        if not img_path.exists():
            continue

        img = Image.open(img_path)
        w, h = img.size

        rows = df[df["image"] == img_name]

        label_file = output_path / f"labels/{split}" / (img_name.replace(".jpg", ".txt"))

        with open(label_file, "w") as f:
            for _, row in rows.iterrows():
                x, y, bw, bh = convert(
                    row["xmin"], row["ymin"],
                    row["xmax"], row["ymax"],
                    w, h
                )

                f.write(f"0 {x} {y} {bw} {bh}\n")

In [9]:
create_labels(train_imgs, "train")
create_labels(val_imgs, "val")

Создаем yaml файл, где указываем пути до train и val выборки, а так же указыавем имеющиеся у нас классы.

In [10]:
yaml_text = """
train: D:/python/II/dataset/images/train
val: D:/python/II/dataset/images/val

names:
  0: car
"""

with open("data.yaml", "w") as f:
    f.write(yaml_text)

Проверяем выборку на битые файлы

In [11]:
labels_path = "dataset/labels/train"

empty_files = []

for f in os.listdir(labels_path):
    path = os.path.join(labels_path, f)
    if os.path.getsize(path) == 0:
        empty_files.append(f)

print("Empty label files:", len(empty_files))
print(empty_files[:10])

Empty label files: 0
[]


In [12]:
from PIL import Image

bad = []

for root, _, files in os.walk("dataset/images"):
    for f in files:
        try:
            Image.open(os.path.join(root, f)).verify()
        except:
            bad.append(f)

print("Bad images:", bad)

Bad images: []


Запускаем обучение. Я тестировал разное количество эпох и в целом количество эпох больше 25+- не дает никакой прибавки. Но для финальной версии я решил оставить 40

In [13]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    workers=0,
    name="yolo11n_exp",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=data.yaml, epochs=40, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=None, name=yolo11n_exp, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save

train: Scanning D:\python\II\dataset\labels\train... 284 images, 0 backgrounds, 0 corrupt: 100%|██████████| 284/284

train: New cache created: D:\python\II\dataset\labels\train.cache



val: Scanning D:\python\II\dataset\labels\val... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 [00:00

val: New cache created: D:\python\II\dataset\labels\val.cache


...

                   all         71        119      0.988      0.941      0.977       0.65



40 epochs completed in 0.126 hours.
Optimizer stripped from runs\detect\yolo11n_exp\weights\last.pt, 5.4MB
Optimizer stripped from runs\detect\yolo11n_exp\weights\best.pt, 5.4MB

Validating runs\detect\yolo11n_exp\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:


                   all         71        119      0.984       0.95      0.974      0.654
Speed: 0.3ms preprocess, 3.7ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\yolo11n_exp


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000290B10D71A0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

Смотрим лучшише метрики:

In [14]:
model = YOLO("runs/detect/yolo11n_exp/weights/best.pt") 

metrics = model.val(data="data.yaml")

results = []

print("\n===== YOLO11n =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

results.append({
    "Model": "YOLO11n",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
})

Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:


                   all         71        119      0.984       0.95      0.974      0.654
Speed: 1.1ms preprocess, 8.5ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs\detect\val

===== YOLO11n =====
Precision: 0.9841
Recall:    0.9496
mAP50:     0.9737
mAP50-95:  0.6545


Высокое значение Precision указывает на то, что модель практически не допускает ложноположительных срабатываний, то есть обнаруженные объекты в большинстве случаев действительно являются автомобилями.

Значение Recall также находится на высоком уровне, что говорит о способности модели находить большинство объектов на изображении. Тем не менее, небольшая разница между Precision и Recall свидетельствует о том, что часть объектов всё же пропускается.

Метрика mAP50 показывает, что при пороге IoU = 0.5 модель демонстрирует очень высокое качество детекции, корректно определяя объекты и их границы в большинстве случаев.

Однако более строгая метрика mAP50-95 значительно ниже. Это связано с тем, что она учитывает качество локализации при разных порогах IoU (от 0.5 до 0.95). Такое снижение указывает на то, что модель не всегда точно определяет границы объектов, особенно при более строгих требованиях к совпадению.

Анализируя метрики, мы можем сделать вывод, что единственное, что нам нужно - пытаться повысить mAP50-95. Остальные метрики уже показывают хорошие результаты. Вот мои гипотезы:

1. Гипотеза: использование более сложной архитектуры модели (YOLO11s вместо YOLO11n) приведёт к увеличению качества локализации объектов за счёт более выразительного представления признаков, что в первую очередь повысит метрику mAP50-95, так как она наиболее чувствительна к точности bounding box при строгих IoU-порогах.

2. Гипотеза: применение усиленных аугментаций данных (mosaic, mixup, цветовые и геометрические преобразования) повысит устойчивость модели к вариативности изображений и улучшит качество обобщения, что приведёт к росту метрики mAP50-95 за счёт более стабильного определения границ объектов в различных условиях.

3. Гипотеза: изменение стратегии дообучения модели через заморозку части слоёв (backbone freezing) на начальных этапах обучения позволит улучшить качество извлечения признаков и стабилизировать обучение, что приведёт к повышению метрики mAP50-95 за счёт более устойчивой адаптации модели к домену данных.

# Для начала проверим гипотезу 1:

In [15]:
model.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    workers=0,
    name="yolo11s_exp",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=runs/detect/yolo11n_exp/weights/best.pt, data=data.yaml, epochs=40, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=None, name=yolo11s_exp, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False,

train: Scanning D:\python\II\dataset\labels\train.cache... 284 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2
val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 

Plotting labels to runs\detect\yolo11s_exp\labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\yolo11s_exp
Starting training for 40 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/40      0.69G      1.531      1.153      1.307         14        640: 100%|██████████| 71/71 [00:10<00:00,
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:

...

                   all         71        119      0.958      0.965      0.977       0.68



40 epochs completed in 0.120 hours.
Optimizer stripped from runs\detect\yolo11s_exp\weights\last.pt, 5.4MB
Optimizer stripped from runs\detect\yolo11s_exp\weights\best.pt, 5.4MB

Validating runs\detect\yolo11s_exp\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:


                   all         71        119      0.958      0.965      0.977      0.681
Speed: 0.3ms preprocess, 2.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\yolo11s_exp


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000029094C6F980>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [16]:
from ultralytics import YOLO

model = YOLO("runs/detect/yolo11s_exp/weights/best.pt")  

metrics = model.val(data="data.yaml")

print("\n===== YOLO11s =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

results.append({
    "Model": "YOLO11s",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
})

Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:


                   all         71        119      0.958      0.965      0.977      0.679
Speed: 1.2ms preprocess, 6.1ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs\detect\val2

===== YOLO11s =====
Precision: 0.9583
Recall:    0.9654
mAP50:     0.9768
mAP50-95:  0.6793


Как можно заметить по новым метрикам практически ничего не изменилось. Модель стала делать чуть больше ложных срабатываний, при этом она стала находить чуть больше объектов на изображении. Ну и наш показатель mAP50-95 тоже чуть вырос, но слишком незначительно, чтобы это можно было считать хоть в какой-нибудь степени успехом. Теперь проверим другие гипотезы. Будем их использовать уже сразу с улучшенной моделью.

# Проверяем гипотезу 2:

In [17]:
model_aug = YOLO("yolo11s.pt")

model_aug.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    workers=0,
    name="yolo11s_aug_exp",
    exist_ok=True,
    mosaic=1.0,
    mixup=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5
)



New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolo11s.pt, data=data.yaml, epochs=40, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=None, name=yolo11s_aug_exp, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, 

train: Scanning D:\python\II\dataset\labels\train.cache... 284 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2
val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 

Plotting labels to runs\detect\yolo11s_aug_exp\labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\yolo11s_aug_exp
Starting training for 40 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/40       1.3G      1.526      2.647      1.245          7        640: 100%|██████████| 71/71 [00:13<00:00,
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:

...

                   all         71        119      0.981      0.958      0.987      0.654



40 epochs completed in 0.151 hours.
Optimizer stripped from runs\detect\yolo11s_aug_exp\weights\last.pt, 19.2MB
Optimizer stripped from runs\detect\yolo11s_aug_exp\weights\best.pt, 19.2MB

Validating runs\detect\yolo11s_aug_exp\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:


                   all         71        119      0.966      0.964      0.987      0.674
Speed: 0.3ms preprocess, 5.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\yolo11s_aug_exp


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000290B09E4A10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [18]:
model = YOLO("runs/detect/yolo11s_aug_exp/weights/best.pt")

metrics = model.val(data="data.yaml")

print("\n===== YOLO11s + AUGMENTATIONS =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

results.append({
    "Model": "YOLO11s + Aug",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
})

Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:


                   all         71        119      0.966      0.964      0.987      0.675
Speed: 1.5ms preprocess, 9.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs\detect\val3

===== YOLO11s + AUGMENTATIONS =====
Precision: 0.9663
Recall:    0.9645
mAP50:     0.9872
mAP50-95:  0.6750


Показатели практически не изменелись по сравнению с тем что было до. Небольшие изменения в большую или меньшую сторону и не более.

# Теперь проверим гипотезу 3:

In [19]:
from ultralytics import YOLO

model_freeze = YOLO("yolo11s.pt")

model_freeze.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    workers=0,
    name="yolo11s_freeze_exp",
    exist_ok=True,
    freeze=10  
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolo11s.pt, data=data.yaml, epochs=40, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=None, name=yolo11s_freeze_exp, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False,

train: Scanning D:\python\II\dataset\labels\train.cache... 284 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2
val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 

Plotting labels to runs\detect\yolo11s_freeze_exp\labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\yolo11s_freeze_exp
Starting training for 40 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/40      1.41G      1.482      2.597      1.201         14        640: 100%|██████████| 71/71 [00:09<00:00,
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:

...

                   all         71        119      0.951      0.958      0.981      0.665



40 epochs completed in 0.122 hours.
Optimizer stripped from runs\detect\yolo11s_freeze_exp\weights\last.pt, 19.2MB
Optimizer stripped from runs\detect\yolo11s_freeze_exp\weights\best.pt, 19.2MB

Validating runs\detect\yolo11s_freeze_exp\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:


                   all         71        119      0.973      0.958      0.987      0.679
Speed: 0.4ms preprocess, 4.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs\detect\yolo11s_freeze_exp


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000290B1515190>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [20]:
from ultralytics import YOLO

model = YOLO("runs/detect/yolo11s_freeze_exp/weights/best.pt")

metrics = model.val(data="data.yaml")

print("\n===== YOLO11s + FREEZE =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

results.append({
    "Model": "YOLO11s + Freeze",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
})

Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:


                   all         71        119      0.973      0.958      0.987      0.681
Speed: 0.9ms preprocess, 8.8ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\val4

===== YOLO11s + FREEZE =====
Precision: 0.9726
Recall:    0.9580
mAP50:     0.9868
mAP50-95:  0.6808


Мы получили наивысшие показатели mAP50-95, но значительной прибавки мы так и не получили. Я пробовал менять значения freeze, но значительных изменений это тоже не дало. Теперь попробуем совместить все 3 гипотезы:

In [21]:
model_combo = YOLO("yolo11s.pt")

model_combo.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,
    batch=4,
    workers=0,
    name="yolo11s_aug_freeze_exp",
    exist_ok=True,

    mosaic=1.0,
    mixup=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,

    freeze=10
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolo11s.pt, data=data.yaml, epochs=40, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=None, name=yolo11s_aug_freeze_exp, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fa

train: Scanning D:\python\II\dataset\labels\train.cache... 284 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2
val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 

Plotting labels to runs\detect\yolo11s_aug_freeze_exp\labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\yolo11s_aug_freeze_exp
Starting training for 40 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/40      1.55G      1.514      2.626      1.234          7        640: 100%|██████████| 71/71 [00:11<00:00,
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:

...

                   all         71        119      0.974       0.95      0.985      0.665



40 epochs completed in 0.126 hours.
Optimizer stripped from runs\detect\yolo11s_aug_freeze_exp\weights\last.pt, 19.2MB
Optimizer stripped from runs\detect\yolo11s_aug_freeze_exp\weights\best.pt, 19.2MB

Validating runs\detect\yolo11s_aug_freeze_exp\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:


                   all         71        119      0.983      0.945      0.984      0.687
Speed: 0.4ms preprocess, 4.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\yolo11s_aug_freeze_exp


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000290B1665AF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [22]:
model = YOLO("runs/detect/yolo11s_aug_freeze_exp/weights/best.pt")
metrics = model.val(data="data.yaml")

print("\n===== YOLO11s + AUG + FREEZE =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

results.append({
    "Model": "YOLO11s + Aug + Freeze",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
})

Ultralytics 8.3.0  Python-3.12.9 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 238 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


val: Scanning D:\python\II\dataset\labels\val.cache... 71 images, 0 backgrounds, 0 corrupt: 100%|██████████| 71/71 
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:


                   all         71        119      0.966      0.966      0.984      0.689
Speed: 1.0ms preprocess, 8.5ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs\detect\val5

===== YOLO11s + AUG + FREEZE =====
Precision: 0.9657
Recall:    0.9664
mAP50:     0.9839
mAP50-95:  0.6890


Сравниваем итоговые результаты:

In [23]:
import pandas as pd

df = pd.DataFrame(results)
df

,Model,Precision,Recall,mAP50,mAP50-95
0,YOLO11n,0.984121,0.949580,0.973670,0.654472
1,YOLO11s,0.958292,0.965390,0.976780,0.679280
2,YOLO11s + Aug,0.966321,0.964450,0.987207,0.675043
3,YOLO11s + Freeze,0.972619,0.957983,0.986845,0.680756
4,YOLO11s + Aug + Freeze,0.965726,0.966387,0.983915,0.689042


Как мы видим совмещение гипотезы 3 и 4 дало наилучшие результаты. Единственное, в чем проигрывает последняя модель - Precision. Все остальные показтели выше чем у других моделей. 

Имплементация собственной модели!

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import os
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# DATASET 
class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=224):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.files = os.listdir(img_dir)
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt"))

        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img / 255.0
        img = torch.tensor(img, dtype=torch.float).permute(2, 0, 1)

        # default: no object
        target = torch.zeros(5)

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                line = f.readline().strip().split()
                if len(line) == 5:
                    _, x, y, w, h = map(float, line)
                    target = torch.tensor([x, y, w, h, 1.0], dtype=torch.float)

        return img, target


train_dataset = YoloDataset("dataset/images/train", "dataset/labels/train")
val_dataset = YoloDataset("dataset/images/val", "dataset/labels/val")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# MODEL
class SimpleYOLO(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.head = nn.Linear(128, 5)  # x,y,w,h,obj

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return torch.sigmoid(self.head(x))


model = SimpleYOLO().to(device)

# LOSS
def loss_fn(pred, target):
    box_loss = F.mse_loss(pred[:, :4], target[:, :4])
    obj_loss = F.binary_cross_entropy(pred[:, 4], target[:, 4])
    return box_loss + obj_loss

# IoU
def iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    b1_x1, b1_y1 = x1 - w1/2, y1 - h1/2
    b1_x2, b1_y2 = x1 + w1/2, y1 + h1/2

    b2_x1, b2_y1 = x2 - w2/2, y2 - h2/2
    b2_x2, b2_y2 = x2 + w2/2, y2 + h2/2

    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area1 = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    area2 = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)

    return inter / (area1 + area2 - inter + 1e-6)

In [25]:
import torch
print(torch.randn(1).cuda())

tensor([0.8591], device='cuda:0')


In [26]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

for epoch in range(40):
    total_loss = 0

    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/40 | Loss: {total_loss:.4f}")

Epoch 1/40 | Loss: 5.9324
Epoch 2/40 | Loss: 1.2309
Epoch 3/40 | Loss: 1.0375
Epoch 4/40 | Loss: 0.9246
Epoch 5/40 | Loss: 0.8999
Epoch 6/40 | Loss: 0.8866
Epoch 7/40 | Loss: 0.9033
Epoch 8/40 | Loss: 0.9101
Epoch 9/40 | Loss: 0.9263
Epoch 10/40 | Loss: 0.9141
Epoch 11/40 | Loss: 0.9445
Epoch 12/40 | Loss: 0.8757
Epoch 13/40 | Loss: 0.9068
Epoch 14/40 | Loss: 0.9278
Epoch 15/40 | Loss: 0.8664
Epoch 16/40 | Loss: 0.8925
Epoch 17/40 | Loss: 0.8623
Epoch 18/40 | Loss: 0.8789
Epoch 19/40 | Loss: 0.8701
Epoch 20/40 | Loss: 0.8697
Epoch 21/40 | Loss: 0.8748
Epoch 22/40 | Loss: 0.8658
Epoch 23/40 | Loss: 0.8742
Epoch 24/40 | Loss: 0.9025
Epoch 25/40 | Loss: 0.8639
Epoch 26/40 | Loss: 0.9075
Epoch 27/40 | Loss: 0.8750
Epoch 28/40 | Loss: 0.8758
Epoch 29/40 | Loss: 0.8750
Epoch 30/40 | Loss: 0.8825
Epoch 31/40 | Loss: 0.8833
Epoch 32/40 | Loss: 0.8656
Epoch 33/40 | Loss: 0.8691
Epoch 34/40 | Loss: 0.8665
Epoch 35/40 | Loss: 0.8556
Epoch 36/40 | Loss: 0.8565
Epoch 37/40 | Loss: 0.8751
Epoch 38/4

In [27]:
def evaluate(model, loader):
    model.eval()

    iou_thresholds = np.linspace(0.5, 0.95, 10)
    aps = []

    with torch.no_grad():
        for t in iou_thresholds:
            tp, fp, fn = 0, 0, 0

            for imgs, targets in loader:
                imgs = imgs.to(device)
                targets = targets.to(device)

                preds = model(imgs)

                for p, gt in zip(preds, targets):
                    if gt[4] == 0:
                        continue

                    iou_score = iou(
                        p[:4].cpu().numpy(),
                        gt[:4].cpu().numpy()
                    )

                    if iou_score >= t:
                        tp += 1
                    else:
                        fp += 1
                        fn += 1

            precision = tp / (tp + fp + 1e-6)
            aps.append(precision)

    mAP50 = aps[0]
    mAP50_95 = np.mean(aps)
    recall = aps[0]

    print("\n===== RESULTS =====")
    print("Precision:", round(aps[0], 4))
    print("Recall:", round(recall, 4))
    print("mAP50:", round(mAP50, 4))
    print("mAP50-95:", round(mAP50_95, 4))

    return {
        "Precision": float(aps[0]),
        "Recall": float(recall),
        "mAP50": float(mAP50),
        "mAP50-95": float(mAP50_95)
    }

In [28]:
metrics = evaluate(model, val_loader)

results.append({
    "Model": "YOLO custom",
    **metrics
})


===== RESULTS =====
Precision: 0.0141
Recall: 0.0141
mAP50: 0.0141
mAP50-95: 0.0028


Ожидаемо самописаня модель показала ужасные результаты. Базовая имплементированная модель имеет крайне ограниченную архитектуру и решает задачу детекции в сильно упрощённой постановке, фактически сводя её к регрессии координат одного объекта на изображении. В модели отсутствуют механизмы, характерные для современных детекторов, такие как якоря, разбиение изображения на сетку или обработка множественных объектов. Из-за этого модель не способна полноценно интерпретировать сцену и корректно локализовать объекты.

Первую гипотезу мы не можем проверить по очевидной причине - мы не можем использовать более мощную модель, ибо ее не существует.

Третья гипотеза также не применяется, поскольку входные данные уже приведены к фиксированному размеру и формату, и изменение разрешения или структуры данных нарушило бы единые условия сравнения между экспериментами. Поэтому дальнейший анализ проводится только для второй гипотезы, как единственного корректного варианта изменения модели без изменения постановки задачи и данных:

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import os
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# DATASET
class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=224):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.files = os.listdir(img_dir)
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt"))

        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img / 255.0
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)

        target = torch.zeros(5)

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                line = f.readline().strip().split()
                if len(line) == 5:
                    _, x, y, w, h = map(float, line)
                    target = torch.tensor([x, y, w, h, 1.0], dtype=torch.float32)

        return img, target


train_dataset = YoloDataset("dataset/images/train", "dataset/labels/train")
val_dataset = YoloDataset("dataset/images/val", "dataset/labels/val")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# MODEL
class SimpleYOLO_H2(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 5)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return torch.sigmoid(self.head(x))


model = SimpleYOLO_H2().to(device)

# LOSS
def loss_fn(pred, target):
    box_loss = F.mse_loss(pred[:, :4], target[:, :4])
    obj_loss = F.binary_cross_entropy(pred[:, 4], target[:, 4])
    return box_loss + obj_loss

# IOU
def iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    b1_x1, b1_y1 = x1 - w1/2, y1 - h1/2
    b1_x2, b1_y2 = x1 + w1/2, y1 + h1/2

    b2_x1, b2_y1 = x2 - w2/2, y2 - h2/2
    b2_x2, b2_y2 = x2 + w2/2, y2 + h2/2

    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area1 = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    area2 = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)

    return inter / (area1 + area2 - inter + 1e-6)

In [30]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

for epoch in range(40):
    total_loss = 0.0

    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/40 | Loss: {total_loss:.4f}")

Epoch 1/40 | Loss: 4.5139
Epoch 2/40 | Loss: 1.1659
Epoch 3/40 | Loss: 1.0466
Epoch 4/40 | Loss: 0.9984
Epoch 5/40 | Loss: 1.0082
Epoch 6/40 | Loss: 0.9313
Epoch 7/40 | Loss: 0.9595
Epoch 8/40 | Loss: 0.9259
Epoch 9/40 | Loss: 0.8772
Epoch 10/40 | Loss: 0.8814
Epoch 11/40 | Loss: 0.8932
Epoch 12/40 | Loss: 0.8622
Epoch 13/40 | Loss: 0.8643
Epoch 14/40 | Loss: 0.8599
Epoch 15/40 | Loss: 0.8954
Epoch 16/40 | Loss: 0.8495
Epoch 17/40 | Loss: 0.8346
Epoch 18/40 | Loss: 0.8994
Epoch 19/40 | Loss: 0.8284
Epoch 20/40 | Loss: 0.8576
Epoch 21/40 | Loss: 0.8348
Epoch 22/40 | Loss: 0.8093
Epoch 23/40 | Loss: 0.8071
Epoch 24/40 | Loss: 0.8207
Epoch 25/40 | Loss: 0.8314
Epoch 26/40 | Loss: 0.8256
Epoch 27/40 | Loss: 0.8622
Epoch 28/40 | Loss: 0.8577
Epoch 29/40 | Loss: 0.8208
Epoch 30/40 | Loss: 0.8233
Epoch 31/40 | Loss: 0.8153
Epoch 32/40 | Loss: 0.8461
Epoch 33/40 | Loss: 0.8587
Epoch 34/40 | Loss: 0.8141
Epoch 35/40 | Loss: 0.8239
Epoch 36/40 | Loss: 0.8288
Epoch 37/40 | Loss: 0.8079
Epoch 38/4

In [31]:
import numpy as np
import torch

def evaluate(model, loader):
    model.eval()

    iou_thresholds = np.linspace(0.5, 0.95, 10)
    aps = []

    with torch.no_grad():
        for t in iou_thresholds:
            tp, fp = 0, 0

            for imgs, targets in loader:
                imgs = imgs.to(device)
                targets = targets.to(device)

                preds = model(imgs)

                for p, gt in zip(preds, targets):
                    if gt[4].item() == 0:
                        continue

                    iou_score = iou(
                        p[:4].detach().cpu().numpy(),
                        gt[:4].detach().cpu().numpy()
                    )

                    if iou_score >= t:
                        tp += 1
                    else:
                        fp += 1

            precision = tp / (tp + fp + 1e-6)
            aps.append(precision)

    mAP50 = aps[0]
    mAP50_95 = float(np.mean(aps))
    recall = aps[0]

    print("\n===== HYPOTHESIS 2 RESULTS =====")
    print("Precision:", round(mAP50, 4))
    print("Recall:", round(recall, 4))
    print("mAP50:", round(mAP50, 4))
    print("mAP50-95:", round(mAP50_95, 4))

    return {
        "Precision": float(mAP50),
        "Recall": float(recall),
        "mAP50": float(mAP50),
        "mAP50-95": float(mAP50_95)
    }

In [32]:
metrics = evaluate(model, val_loader)

results.append({
    "Model": "SimpleYOLO + Hypothesis 2",
    **metrics
})


===== HYPOTHESIS 2 RESULTS =====
Precision: 0.0845
Recall: 0.0845
mAP50: 0.0845
mAP50-95: 0.0324


Во второй конфигурации модель была усложнена за счёт увеличения глубины сверточной части сети, что теоретически должно улучшать способность извлекать признаки. И мы действительно увидели небольшое улучшение.

# Смотрим полный итоговые результаты:

In [33]:
import pandas as pd

df = pd.DataFrame(results)
df

,Model,Precision,Recall,mAP50,mAP50-95
0,YOLO11n,0.984121,0.949580,0.973670,0.654472
1,YOLO11s,0.958292,0.965390,0.976780,0.679280
2,YOLO11s + Aug,0.966321,0.964450,0.987207,0.675043
3,YOLO11s + Freeze,0.972619,0.957983,0.986845,0.680756
4,YOLO11s + Aug + Freeze,0.965726,0.966387,0.983915,0.689042
5,YOLO custom,0.014085,0.014085,0.014085,0.002817
6,SimpleYOLO + Hypothesis 2,0.084507,0.084507,0.084507,0.032394


# Заключение

В ходе экспериментов с YOLO11 были получены результаты для нескольких конфигураций модели, включая базовые версии и варианты с применением техник улучшенного бейзлайна. В целом можно отметить, что все версии YOLO демонстрируют высокое качество детекции по метрикам Precision, Recall и mAP50, что говорит о стабильной работе модели на данном наборе данных и хорошей способности находить и классифицировать объекты.

При сравнении различных конфигураций видно, что базовая модель YOLO11n уже обеспечивает достаточно высокий уровень качества, однако переход к более крупной версии YOLO11s даёт небольшой прирост по большинству метрик, что подтверждает влияние увеличения ёмкости модели на качество детекции.

Применение аугментаций и заморозки части слоёв приводит к незначительным колебаниям результатов, при этом наиболее стабильные и сбалансированные показатели наблюдаются у комбинации этих подходов. В этой конфигурации достигается наилучшее значение метрики mAP50–95, что указывает на более качественную локализацию объектов при разных порогах IoU и более устойчивую работу модели в сложных случаях.

Таким образом, можно сделать вывод, что основное улучшение качества достигается не за счёт одной конкретной техники, а за счёт их комбинирования, при этом эффект от отдельных модификаций является умеренным. Модель YOLO в целом показывает высокую устойчивость к изменениям конфигурации, а улучшения носят инкрементальный характер, а не радикальный.